# GitHub作業用 Colab ノートブック
GitHubから導入する重いツール（音声解析・高解像度化・TTSなど）を、ColabのGPUで試すための作業ノート。

**最初に必ず**：メニュー「ランタイム」→「ランタイムのタイプを変更」→ ハードウェア アクセラレータで **T4 GPU** を選んで保存。

**使い分けのルール**
- 重い・単発の処理 → このColab（GPU）
- 毎晩の自動制作で無人で使うもの → ローカル環境（従来どおり）
- Googleアカウントに上げる素材は**健全なものだけ**。NSFW寄りの素材はローカルで処理する。

## ① ファイルのアップロード
処理したい音声（wav/mp3）や画像をアップロードする。アップロードしたファイル名に合わせて、下のセル内の `input.wav` / `input.png` の名前を書き換える。

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

## ② WhisperX（単語単位の時刻つき文字起こし）
テロップ同期の精度上げ＋ピッチ解析用。初回はモデルのダウンロードで数分かかる。

In [ ]:
!pip -q install whisperx
import whisperx, json

device = "cuda"
model = whisperx.load_model("small", device, compute_type="float16", language="ja")
audio = whisperx.load_audio("/content/input.wav")
result = model.transcribe(audio, batch_size=16)
align_model, metadata = whisperx.load_align_model(language_code="ja", device=device)
result = whisperx.align(result["segments"], align_model, metadata, audio, device)

with open("/content/whisperx_result.json", "w", encoding="utf-8") as f:
    json.dump(result["segments"], f, ensure_ascii=False, indent=2)
print("OK: /content/whisperx_result.json")
files.download("/content/whisperx_result.json")

## ③ Real-ESRGAN（画像の4倍高解像度化）
512×896のような小さい静止画を 2048×3584 に拡大する。健全な静止画専用（ルール参照）。

In [ ]:
!git clone -q https://github.com/xinntao/Real-ESRGAN
%cd Real-ESRGAN
!pip -q install basicsr facexlib gfpgan
!pip -q install -r requirements.txt
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights
!python inference_realesrgan.py -n RealESRGAN_x4plus -i /content/input.png -o /content/results --outscale 4
print("OK: /content/results")

## ④ Qwen3-TTS（声クローン）— 次に試す用
なぎ本人の声を使ったクローン再演の検証用。枠だけ用意してある。

In [ ]:
# 検証時にここへ実装する
# !pip -q install qwen-tts などを想定
print("TODO")

## ⑤ LatentSync（音声に合わせて口パクを作り直す）
声クローン（④Qwen3-TTS）で作った音声に、映像の口元を合わせ直すためのもの。
入力：元の動画 `input_video.mp4` と、合わせたい音声 `input_audio.wav`。
初回はモデルのダウンロードで時間がかかる。チェックポイントの置き場所でエラーが出たら、LatentSync公式READMEの手順に合わせてこのセルを調整する。

In [ ]:
!git clone -q https://github.com/bytedance/LatentSync
%cd LatentSync
!pip -q install -r requirements.txt
!huggingface-cli download ByteDance/LatentSync-1.5 latentsync_unet.pt --local-dir checkpoints
!huggingface-cli download ByteDance/LatentSync-1.5 whisper/tiny.pt --local-dir checkpoints
!python -m scripts.inference --unet_config_path configs/unet/stage2.yaml --inference_ckpt_path checkpoints/latentsync_unet.pt --inference_steps 20 --video_path /content/input_video.mp4 --audio_path /content/input_audio.wav --video_out_path /content/lipsync_out.mp4
print("OK: /content/lipsync_out.mp4")

## ⑥ CodeFormer（顔の修復）
劣化・圧縮で荒れた顔を復元する。③Real-ESRGANの拡大と組み合わせると「小さい＋荒い」素材の底上げになる。
`-w` は修復の強さ（0〜1）。上げすぎると別人寄りになるので 0.5〜0.7 で試す。

In [ ]:
!git clone -q https://github.com/sczhou/CodeFormer
%cd CodeFormer
!pip -q install -r requirements.txt
!wget -q https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth -P weights/CodeFormer
!python inference_codeformer.py -w 0.7 --input_path /content/input.png --output_path /content/results
print("OK: /content/results")

## ⑦ GPT-SoVITS（声クローンTTS）

なぎの綺麗な自作テイク（5〜10秒）を参照音声にして、別の台詞をその声で読み直すための検証用。④Qwen3-TTSと並ぶ「クローン再演」ルートの本命候補（★65,000、日本語の実績が厚い）。
入力：参照音声 `ref.wav`（①でアップロード）と、その音声の正確な書き起こしテキスト。下のセルの `prompt_text` に書き起こし、`text` に読ませたい台詞を入れる。


In [ ]:
# セットアップ（初回のみ数分かかる）
!git clone -q https://github.com/RVC-Boss/GPT-SoVITS
%cd GPT-SoVITS
!pip -q install -r requirements.txt
!huggingface-cli download lj1995/GPT-SoVITS --local-dir GPT_SoVITS/pretrained_models

# 推論（ゼロショット・クローン）
import sys
sys.path.append('GPT_SoVITS')
from TTS_infer_pack import TTS, TTS_Config
import soundfile as sf

tts = TTS(TTS_Config())
req = {
    "text": "ここに読ませたい台詞",          # 読ませたい台詞
    "text_lang": "ja",
    "ref_audio_path": "/content/ref.wav",  # 参照音声（①でアップロード）
    "prompt_text": "参照音声の書き起こし",     # 参照音声で実際に喋っている内容を正確に
    "prompt_lang": "ja",
    "streaming_mode": False,
}
for sr, audio in tts.run(req):
    sf.write('/content/clone_out.wav', audio, sr)
print('saved: /content/clone_out.wav')
# ※ 版によって TTS_infer_pack の置き場所や引数が変わることがある。その時はリポジトリのREADMEに合わせる。


## ⑧ InsightFace（顔の同一性スコア）

基準顔と生成フレームの顔が「同じ人物か」を数値（-1〜1）で測る。監査チェックの「顔の保持」を目視だけでなく数字で裏付ける用。
入力：基準顔 `ref.jpg` と比較したい画像 `target.jpg`（このセルの中で直接アップロードできる）。
目安：同じ人物なら概ね0.4以上に出ることが多いが、最終的には手元のテイクで基準を調整する。


In [ ]:
!pip -q install insightface onnxruntime-gpu
from google.colab import files
import cv2, numpy as np
from insightface.app import FaceAnalysis

print('基準顔（ref.jpg）を選択')
up = files.upload(); ref_path = list(up.keys())[0]
print('比較画像（target.jpg）を選択')
up = files.upload(); tgt_path = list(up.keys())[0]

app = FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
app.prepare(ctx_id=0, det_size=(640, 640))

def embedding(path):
    faces = app.get(cv2.imread(path))
    assert faces, f'顔が検出できませんでした: {path}'
    return faces[0].normed_embedding

score = float(embedding(ref_path) @ embedding(tgt_path))
print(f'類似度スコア: {score:.3f}  （1に近いほど同一人物）')


## ⑨ SeedVR2（動画の超解像）

動画版の高解像度化。Floyo出力（512×896）のような小さい動画の底上げ用で、③Real-ESRGANの動画版という位置づけ。
入力：動画 `input.mp4`（①でアップロード）。重い処理なので、まず数秒の短いクリップで試すこと。


In [ ]:
# セットアップ
!git clone -q https://github.com/numz/SeedVR2 seedvr2
%cd seedvr2
!pip -q install -r requirements.txt

# 実行（入力→出力）
!python seedvr2.py --video_path /content/input.mp4 --output_path /content/output_up.mp4
print('saved: /content/output_up.mp4')
# ※ CLIの引数名は版で変わることがある。エラーが出たらリポジトリのREADMEのUsageに合わせる。
